# Day 1 Lab: Economics of FinTech — Simulation Exercises

Interactive exploration of cost structures, platform dynamics, and
prediction economics

> **Expected Time**
>
> -   Exercise 1 (Structured case analysis — no code): ≈ 25 minutes
> -   Exercise 2 (Platform simulation): ≈ 40 minutes
> -   Exercise 3 (Effective fee calculator): ≈ 25 minutes
> -   Total: ≈ 90 minutes

<figure>
<a
href="https://colab.research.google.com/github/quinfer/minimba-notebooks/blob/main/lab01_economics.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Setup (Colab-only installs)

## Before You Start

This lab reinforces the four frameworks from today’s lectures. You do
**not** need to be a programmer to complete it. The code is provided and
runs in Google Colab; your job is to **interpret results, change
parameters, and connect findings to theory**.

The three exercises are designed in ascending order of coding
involvement:

1.  **No code** — Structured case analysis applying the economics of
    prediction
2.  **Guided simulation** — Change parameters, observe and interpret
    results
3.  **Guided computation** — Calculate and visualise fee economics

Each exercise has clear deliverables: short written interpretations
(150–300 words) that connect observations to economic theory. These
prepare you for the final assessment (briefing note).

# Exercise 1 — Economics of Prediction: Decompose a Financial Service (25 min)

## The Framework

Recall from this morning’s lecture the “anatomy of a task” (Agrawal,
Gans, and Goldfarb 2018). Every business decision can be decomposed
into:

| Component | Definition | AI impact |
|------------------------|------------------------|------------------------|
| **Prediction** | Forecasting outcomes from data and patterns | AI makes this cheap |
| **Judgement** | Determining relative payoffs, including for errors | Value *increases* as prediction cheapens |
| **Action** | Executing the chosen course | Can be automated for clear cases |
| **Data** | Input, training, and feedback information | More available, cheaper to collect |

As prediction becomes commoditised, the scarce and valuable components
are **judgement** (knowing what matters), **data curation** (knowing
what to feed the model), and **thoughtful action** (knowing when to
override).

## Your Task (No Code Required)

Choose **one** of the following financial services. For your chosen
service, complete the decomposition table below.

**Option A: Robo-advisory (e.g., Nutmeg, Wealthify, Vanguard PAS)**

**Option B: Fraud detection in card payments (e.g., Visa’s real-time
scoring)**

**Option C: Mortgage underwriting (e.g., a digital lender like Habito or
Better)**

**Option D: AML/KYC compliance screening (e.g., checking new customers
against sanctions lists)**

### Decomposition Template

Copy this table and fill it in for your chosen service:

| Component | What does this involve in your chosen service? | Who/what does it today? | Could AI improve it? | What happens if AI gets it wrong? |
|-------|------------------------|-------------|------------|------------------|
| **Prediction** |  |  |  |  |
| **Judgement** |  |  |  |  |
| **Action** |  |  |  |  |
| **Data** |  |  |  |  |

### Interpretation Questions (Write 200–300 words)

1.  **Where is the bottleneck?** Is the service expensive because
    prediction is hard, because judgement is complex, or because data is
    scarce?

2.  **Where does AI help most?** Which component benefits most from
    cheaper prediction? Which component is hardest to automate?

3.  **What is the cost of error?** If the AI prediction is wrong, what
    is the consequence? Is the consequence symmetric (equally bad in
    both directions) or asymmetric (worse in one direction)?

4.  **What automation level is appropriate?** Using the graduated
    automation scale from this morning (Level 0 = fully manual, Level 3
    = fully automated), what level would you recommend and why?

> **Connection to Day 1 Frameworks**
>
> This exercise applies the **economics of prediction** (Part II) and
> the **genuine innovation vs arbitrage** distinction (Part IV). A
> service that automates prediction where prediction is genuinely the
> bottleneck is genuine innovation. A service that automates prediction
> where judgement is the bottleneck is likely overselling AI.

# Exercise 2 — Platform Adoption Simulation (40 min)

## The Model

Two-sided platforms (card networks, payment apps, open banking) obey
coupled dynamics: growth on each side depends on the other side’s
participation, pricing, and governance quality.

The model captures four forces:

$$\text{Growth} = \underbrace{\text{Baseline}}_{\text{marketing}} + \underbrace{\text{Network effect}}_{\text{cross-side value}} - \underbrace{\text{Price friction}}_{\text{fees}} - \underbrace{\text{Congestion}}_{\text{same-side problems}}$$

Formally:

$$N_u(t+1) = N_u(t) + a_u \cdot (1 - N_u) \cdot \max(0, 1 + \beta_{um} \cdot N_m - p_u) - \gamma_u \cdot N_u^2$$

Don’t worry about the maths — focus on the four forces and what happens
when you change them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_platform(T=60, a_u=0.02, a_m=0.015,
                      beta_um=0.6, beta_mu=0.5,
                      price_u=0.0, price_m=0.03,
                      gamma_u=0.0, gamma_m=0.0):
    """
    Simulate two-sided platform adoption.
    
    Parameters:
    - T: time periods (e.g., months)
    - a_u, a_m: baseline growth rates (users, merchants)
    - beta_um, beta_mu: cross-side network effect strengths
    - price_u, price_m: fees/frictions on each side
    - gamma_u, gamma_m: same-side congestion
    
    Returns: (user_participation, merchant_participation) arrays
    """
    Nu = np.zeros(T + 1)
    Nm = np.zeros(T + 1)
    Nu[0] = 0.02
    Nm[0] = 0.02
    
    for t in range(T):
        du = a_u * (1 - Nu[t]) * max(0.0, (1 + beta_um * Nm[t] - price_u)) - gamma_u * Nu[t]**2
        dm = a_m * (1 - Nm[t]) * max(0.0, (1 + beta_mu * Nu[t] - price_m)) - gamma_m * Nm[t]**2
        Nu[t + 1] = np.clip(Nu[t] + du, 0, 1)
        Nm[t + 1] = np.clip(Nm[t] + dm, 0, 1)
    
    return Nu, Nm

print("Simulation function loaded successfully.")

## Task 2a — Four Pricing Scenarios

Run the simulation under four strategies and compare adoption
trajectories.

In [ ]:
scenarios = {
    'Merchant-pays (baseline)': dict(price_u=0.0, price_m=0.03),
    'User-pays (wrong side)':   dict(price_u=0.03, price_m=0.0),
    'Both free (no revenue)':   dict(price_u=0.0, price_m=0.0),
    'Congestion (poor governance)': dict(price_u=0.0, price_m=0.03, gamma_u=0.15, gamma_m=0.10),
}

plt.figure(figsize=(12, 6))
for label, params in scenarios.items():
    Nu, Nm = simulate_platform(**params)
    plt.plot(Nu, label=f'Users — {label}', linewidth=1.8)
    plt.plot(Nm, linestyle='--', label=f'Merchants — {label}', linewidth=1.8, alpha=0.7)

plt.title('Platform Adoption Under Different Pricing Strategies')
plt.xlabel('Time Period')
plt.ylabel('Participation (0 = none, 1 = full market)')
plt.grid(alpha=0.3, linestyle=':')
plt.legend(ncol=2, fontsize=9, loc='right')
plt.tight_layout()
plt.show()

### Questions (Write 150–200 words)

1.  Why does the “user-pays” scenario perform so much worse than
    “merchant-pays”? (Hint: which side is more price-sensitive? Which
    side drives stronger cross-side effects?)
2.  The “both free” scenario reaches the highest adoption — so why
    doesn’t every platform do this?
3.  The “congestion” scenario has the same pricing as baseline but lower
    adoption. What real-world problems does congestion represent?
    (Think: fraud, low-quality listings, spam, API abuse.)

## Task 2b — Price Sensitivity Heatmap

Now systematically explore how different combinations of user and
merchant prices affect adoption.

In [ ]:
price_grid = np.linspace(0.0, 0.06, 13)
Z = np.zeros((len(price_grid), len(price_grid)))

for i, pu in enumerate(price_grid):
    for j, pm in enumerate(price_grid):
        Nu, Nm = simulate_platform(price_u=pu, price_m=pm)
        Z[i, j] = Nu[-1]

plt.figure(figsize=(8, 6))
im = plt.imshow(Z, origin='lower',
                extent=[0, 0.06, 0, 0.06],
                aspect='auto', cmap='viridis')
plt.colorbar(im, label='Final user participation')
plt.xlabel('Merchant price (p_m)')
plt.ylabel('User price (p_u)')
plt.title('User Adoption vs Price Structure')
plt.tight_layout()
plt.show()

### Key Observation

The heatmap is **asymmetric**: increasing user price (moving up)
destroys adoption far more than increasing merchant price (moving
right). This is why consumer-facing FinTech is typically free.

### Questions (Write 150–200 words)

1.  Where is the “sweet spot” — the region that maximises adoption while
    still generating some revenue?
2.  A regulator caps merchant fees at 2% (like EU interchange caps).
    Using the heatmap, predict what happens to adoption. What if the
    platform responds by introducing a small user fee?
3.  How does this connect to Day 1’s cost puzzle? If platforms charge
    merchants to subsidise users, where does the “cost” go?

## Task 2c — Network Effect Strength

How strong do network effects need to be for a platform to succeed?

In [ ]:
betas = np.linspace(0.3, 0.9, 13)
finals_users = []
finals_merchants = []

for b in betas:
    Nu, Nm = simulate_platform(beta_um=b, beta_mu=0.5)
    finals_users.append(Nu[-1])
    finals_merchants.append(Nm[-1])

plt.figure(figsize=(10, 5))
plt.plot(betas, finals_users, 'o-', linewidth=2, label='User participation')
plt.plot(betas, finals_merchants, 's--', linewidth=2, label='Merchant participation')
plt.xlabel('Network effect strength (β_um)')
plt.ylabel('Final participation')
plt.title('How Strong Must Network Effects Be for Take-Off?')
plt.legend()
plt.grid(alpha=0.3, linestyle=':')
plt.tight_layout()
plt.show()

### Questions (Write 100–150 words)

1.  At roughly what value of $\beta$ does the platform “take off”? What
    does this mean for a new entrant trying to challenge an established
    platform?
2.  For open banking, $\beta_{um}$ represents how much app developers
    value bank API coverage. If developers don’t care much (low
    $\beta$), what does that imply for adoption?

# Exercise 3 — The Economics of Automation: Fee Structures (25 min)

## Why This Matters

The economics of prediction framework says: when prediction becomes
cheap, the value shifts to judgement. Robo-advisors are the clearest
financial example — they automate prediction (portfolio optimisation,
rebalancing) while leaving judgement (risk tolerance, life goals) to the
client or a human advisor.

But the *cost structure* reveals something additional: traditional
advisors impose minimum fees that make small accounts prohibitively
expensive. Automation doesn’t just make advice cheaper — it **expands
access** to people who were previously excluded.

## Task 3a — Effective Fee Rates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

account_sizes = np.array([10_000, 25_000, 50_000, 100_000, 250_000, 500_000])

# Traditional adviser: 1.5% AUM with £2,500 minimum
trad_rate = 0.015
trad_min = 2500
trad_fee = np.maximum(trad_rate * account_sizes, trad_min)

# Robo-adviser: 0.25% AUM, no minimum
robo_rate = 0.0025
robo_fee = robo_rate * account_sizes

# Effective rates (what you actually pay as % of assets)
trad_effective = trad_fee / account_sizes * 100
robo_effective = robo_fee / account_sizes * 100

# Annual savings
savings = trad_fee - robo_fee

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Effective fee rate
axes[0].plot(account_sizes / 1000, trad_effective, 'o-', linewidth=2,
             label='Traditional (1.5% + £2,500 min)', color='tab:blue')
axes[0].plot(account_sizes / 1000, robo_effective, 's-', linewidth=2,
             label='Robo (0.25%, no min)', color='tab:red')
axes[0].set_xlabel('Account Size (£000s)')
axes[0].set_ylabel('Effective Fee Rate (%)')
axes[0].set_title('Binding Minimums Raise Effective Fees on Small Accounts')
axes[0].legend()
axes[0].grid(alpha=0.3, linestyle=':')

# Panel 2: Annual savings
axes[1].bar([f'£{s//1000:.0f}k' for s in account_sizes], savings, color='tab:green')
for i, v in enumerate(savings):
    axes[1].text(i, v + 50, f'£{v:,.0f}', ha='center', fontsize=9)
axes[1].set_ylabel('Annual Savings (£)')
axes[1].set_title('Annual Savings from Robo vs Traditional')
axes[1].grid(axis='y', alpha=0.3, linestyle=':')

plt.tight_layout()
plt.show()

print(f"At £10k: Traditional charges £{trad_fee[0]:,.0f} "
      f"(effective {trad_effective[0]:.1f}%), Robo charges £{robo_fee[0]:,.0f} "
      f"(effective {robo_effective[0]:.2f}%)")
print(f"At £500k: Traditional charges £{trad_fee[-1]:,.0f}, Robo charges £{robo_fee[-1]:,.0f}")

### Questions (Write 150–200 words)

1.  For a £10,000 account, the traditional advisor charges an effective
    rate of 25%. Why does this happen? Why don’t traditional advisors
    simply drop their minimums?

2.  At what account size does the traditional advisor become competitive
    with the robo-advisor? What does this tell you about who benefits
    most from automation?

3.  Connect this to the economics of prediction: the robo-advisor
    automates *prediction* (portfolio allocation). The traditional
    advisor provides *judgement* (complex tax planning, estate advice,
    behavioural coaching). At what wealth level does the value of that
    human judgement justify the higher fee?

## Task 3b — Compounding the Savings (Optional Extension)

If you invest the fee savings at 7% annual return, how much extra wealth
do you accumulate over 20 years?

In [ ]:
years = 20
annual_return = 0.07
account = 50_000  # £50k example

trad_annual = max(trad_rate * account, trad_min)
robo_annual = robo_rate * account
annual_saving = trad_annual - robo_annual

# Compound the savings (invest each year's saving)
cumulative = 0
trajectory = []
for y in range(years):
    cumulative = (cumulative + annual_saving) * (1 + annual_return)
    trajectory.append(cumulative)

plt.figure(figsize=(9, 5))
plt.plot(range(1, years + 1), trajectory, 'o-', linewidth=2, color='tab:green')
plt.xlabel('Year')
plt.ylabel('Cumulative Extra Wealth (£)')
plt.title(f'Compounding Fee Savings: £{account//1000}k Account Over {years} Years')
plt.grid(alpha=0.3, linestyle=':')
for y in [5, 10, 15, 20]:
    plt.annotate(f'£{trajectory[y-1]:,.0f}', (y, trajectory[y-1]),
                 textcoords='offset points', xytext=(10, 5), fontsize=9)
plt.tight_layout()
plt.show()

print(f"Annual saving: £{annual_saving:,.0f}")
print(f"After {years} years (invested at {annual_return:.0%}): £{trajectory[-1]:,.0f} extra wealth")

This makes the cost puzzle concrete: the £625 annual saving on a £50,000
account compounds to over £25,000 in extra wealth over 20 years. Small
fee differences have large long-run consequences.

# Synthesis — Connecting the Exercises

## The Through-Line

These three exercises illustrate a single argument:

1.  **Exercise 1** (prediction decomposition): Identified *where* in a
    financial service AI can reduce costs and *where* human judgement
    remains essential.

2.  **Exercise 2** (platform simulation): Showed *how* platform dynamics
    determine which business models succeed — pricing structure matters
    more than pricing level, and governance matters as much as pricing.

3.  **Exercise 3** (fee economics): Demonstrated *who benefits* when
    automation reduces costs — and that the largest gains accrue to
    previously excluded participants (small accounts).

Together, these exercises operationalise Day 1’s frameworks: the cost
puzzle (Exercise 3 shows where costs compress), the economics of
prediction (Exercise 1 decomposes the value chain), platform economics
(Exercise 2 shows scaling dynamics), and the evidence lens (all
exercises require you to interpret results critically, not just accept
outputs).

## Final Reflection (200–300 words)

Write a short reflection connecting your findings across all three
exercises:

1.  Which financial services are most susceptible to FinTech disruption?
    Why? (Use your Exercise 1 decomposition and Exercise 3 cost
    analysis.)
2.  What role do platform dynamics play in determining which FinTech
    innovations succeed? (Use your Exercise 2 simulation findings.)
3.  Where is the line between genuine innovation and regulatory
    arbitrage? How would you tell the difference?

This reflection is direct preparation for the final assessment (briefing
note).

## References

Agrawal, Ajay, Joshua Gans, and Avi Goldfarb. 2018. *Prediction
Machines: The Simple Economics of Artificial Intelligence*. Harvard
Business Review Press.